# L66 · ASR 系统全览 — 声学模型 → 语言模型 → 解码器，端到端 vs 经典流水线

**目标**：建立现代端到端 ASR 的完整系统观——特征提取 → 编码器 → 解码器 → 文字，理解各环节的输入/输出形状与作用。

🔗 **Aurora 连接**：`aurora.audio.mel` 实现了 ASR 的特征提取层（log-Mel 频谱）；`aurora.speech`（本月新建）将承载 CTC/Attention 编解码器。

现代 ASR 把一段连续波形变成离散词序列。端到端模型（Whisper、wav2vec 2.0）把原本分开优化的声学模型、语言模型合并成一个大网络，唯一的监督信号是配对的（音频, 文字）样本。理解三个支柱——**特征**、**序列建模**、**评估指标**——是本月的主线。

In [ ]:
import numpy as np

## 1. 端到端 ASR vs 传统 HMM-GMM

传统流水线把任务切成三段独立训练：**声学模型**（GMM 建模帧概率）→ **发音词典** → **语言模型**（N-gram）。误差无法端到端反传，各段最优不代表整体最优。

端到端系统（Whisper/wav2vec 2.0）直接学 P(文字 | 音频)：Transformer 编码器处理帧序列，解码器（或 CTC 头）输出 token 概率。一次梯度下降同时优化全链路，数据量够大时效果碾压传统方案。

架构对比（简化）：
```
HMM-GMM: waveform → MFCC → GMM/HMM → 词典 → N-gram LM → text
Whisper:  waveform → log-Mel → Transformer Enc → Transformer Dec → text
```

In [ ]:
# 演示：两种路径的「接口」差异
# 传统方案输出每帧的 HMM 状态后验概率；端到端直接输出 token logits

T, n_mels = 300, 80   # 300 帧，80维 log-Mel
n_vocab   = 50257     # Whisper 词表大小

# 传统：每帧 → GMM 状态概率（假设 3000 个 HMM 状态）
n_hmm_states = 3000
hmm_output_shape = (T, n_hmm_states)

# 端到端：编码器输出序列 → 解码器输出 token 概率
enc_output_shape = (T // 2, 512)   # 2× 下采样后，d_model=512
dec_output_shape = (n_vocab,)       # 每步预测一个 token

print(f'HMM-GMM 每帧输出维度: {hmm_output_shape}')
print(f'Whisper 编码器输出形状: {enc_output_shape}')
print(f'Whisper 解码器每步输出维度: {dec_output_shape}')

## 2. 特征：80-dim log-Mel 频谱

Whisper 的标准输入是 **80维 log-Mel 频谱**，帧参数与 Week 3 完全一致：
```
采样率   sr = 16000 Hz
窗长     n_fft  = 400  样本  （25 ms）
帧移     hop    = 160  样本  （10 ms）
Mel bins n_mels = 80
```

处理流程：`stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))`。结果形状为 `(80, T_frames)`，T_frames ≈ 总样本数 / hop。

为什么 Mel 而非线性频率？人耳对低频分辨率更高，Mel 尺度模拟这一非线性，让模型更容易学到音素边界。

In [ ]:
# 演示：计算一段随机信号的 log-Mel 频谱形状
sr     = 16000
n_fft  = 400
hop    = 160
n_mels = 80
duration_s = 3.0

n_samples  = int(sr * duration_s)
n_frames   = 1 + (n_samples - n_fft) // hop

print(f'音频长度: {duration_s} s  →  {n_samples} 样本')
print(f'STFT 帧数 (n_fft=400, hop=160): {n_frames}')
print(f'log-Mel 频谱形状: ({n_mels}, {n_frames})')
print(f'每帧覆盖: {n_fft/sr*1000:.1f} ms，帧移: {hop/sr*1000:.1f} ms')

## 3. WER — 词错率

WER（Word Error Rate）是 ASR 最核心的评估指标：
```
WER = (S + D + I) / N
```
- `N`：参考（正确）文本的总词数
- `S`（Substitution）：替换错误数——模型输出了错词
- `D`（Deletion）：删除错误数——漏说了某词
- `I`（Insertion）：插入错误数——多说了某词

WER 通过**编辑距离**（Levenshtein distance）计算，以词为单位而非字符。WER 可以 > 1.0（大量插入时）。

常见基线（英语）：
```
Whisper-large-v3    WER ≈ 2–5%   (LibriSpeech test-clean)
Whisper-base        WER ≈ 5–8%
传统 HMM-GMM        WER ≈ 8–15%
```

In [ ]:
# 演示：手工追踪一个 WER 计算例子
reference  = ['the', 'cat', 'sat', 'on', 'the', 'mat']
hypothesis = ['the', 'cat', 'sat', 'on', 'a',   'mat']
#                                           ^ 替换: the→a

# 手工统计
S = 1   # 'the' → 'a'
D = 0
I = 0
N = len(reference)
wer_manual = (S + D + I) / N

print(f'参考: {reference}')
print(f'假设: {hypothesis}')
print(f'S={S}, D={D}, I={I}, N={N}')
print(f'WER = ({S}+{D}+{I})/{N} = {wer_manual:.4f} = {wer_manual*100:.2f}%')

## 4. ✏️ 实现 `compute_wer(hypothesis, reference)`

**推理路线**：
1. 用动态规划构建编辑距离矩阵：`dp[i][j]` 表示把 `hyp[:i]` 变成 `ref[:j]` 所需的最少操作数（替换/删除/插入各计 1 次）
2. 状态转移：若 `hyp[i-1] == ref[j-1]` 则 `dp[i][j] = dp[i-1][j-1]`；否则 `dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])`
3. 总错误数 = `dp[len(hyp)][len(ref)]`，除以 `len(reference)` 得到 WER

**参考输入输出**：
```
hyp = ['the', 'cat', 'sat']
ref = ['the', 'sat', 'cat']
编辑距离 = 2（交换 cat/sat 需要1次替换+1次替换）
WER = 2 / 3 ≈ 0.6667
```

<details><summary>点击查看参考实现</summary>

```python
def compute_wer(hypothesis: list, reference: list) -> float:
    H, R = len(hypothesis), len(reference)
    if R == 0:
        return 0.0 if H == 0 else float('inf')
    # 初始化 DP 表
    dp = list(range(R + 1))
    for i in range(1, H + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, R + 1):
            temp = dp[j]
            if hypothesis[i - 1] == reference[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = temp
    return dp[R] / R
```

</details>

In [ ]:
def compute_wer(hypothesis: list, reference: list) -> float:
    """
    计算词错率 WER = (S+D+I) / len(reference)。
    hypothesis: 模型输出的词列表
    reference:  正确文本的词列表
    """
    H, R = len(hypothesis), len(reference)
    # ✏️ TODO: 边界条件

    # ✏️ TODO: 初始化 dp 数组（长度 R+1，表示把空序列变成 ref[:j] 的代价）

    # ✏️ TODO: 遍历 hypothesis 的每个词，更新 dp（滚动数组，O(R) 空间）

    # ✏️ TODO: return dp[R] / R
    pass

In [ ]:
# 检查 compute_wer
assert abs(compute_wer(['the', 'cat'], ['the', 'cat']) - 0.0) < 0.001, '完全一致 WER 应为 0'
assert abs(compute_wer([], ['a']) - 1.0) < 0.001, '全删除 WER 应为 1'
assert abs(compute_wer(['the', 'cat', 'sat'], ['the', 'sat', 'cat']) - 2/3) < 0.001, \
    '交换两词 WER 应为 2/3'
assert abs(compute_wer(['a', 'b', 'c', 'd'], ['b', 'c']) - 1.0) < 0.001, \
    '多余词导致插入错误'
print('✅ compute_wer 全部检查通过')

## 5. 参数实验：错误类型分析

模拟 10 句话的 ASR 转写结果，统计 S/D/I 分布，观察不同类型错误对 WER 的贡献。

**预期现象**：
- 背景噪声场景：**D（删除）** 占比高，弱读音节被吞掉
- 口音场景：**S（替换）** 占比高，音素混淆
- 语速极快场景：**D** 再次升高，词边界丢失

实验中修改 `pairs` 列表，替换成自己的转写结果，观察统计变化。

In [ ]:
# 模拟 10 句话的参考文本与 ASR 假设（手工输入或真实转写）
pairs = [
    # (hypothesis_words, reference_words)
    (['hello', 'world'],                    ['hello', 'world']),           # 完美
    (['the', 'cat', 'sat'],                 ['the', 'cat', 'sat', 'down']),# 删除
    (['i', 'love', 'music'],                ['i', 'love', 'music']),       # 完美
    (['this', 'is', 'a', 'test'],           ['this', 'is', 'test']),       # 插入
    (['speech', 'recognition'],             ['speech', 'recognition']),    # 完美
    (['the', 'whether', 'is', 'nice'],      ['the', 'weather', 'is', 'nice']),# 替换
    (['aurora', 'audio'],                   ['aurora', 'audio', 'ai']),    # 删除
    (['deep', 'learning', 'rocks'],         ['deep', 'learning', 'rocks']),# 完美
    (['transformer', 'model'],              ['transformer', 'models']),    # 替换
    (['end', 'to', 'end', 'system'],        ['end', 'to', 'end', 'system']),# 完美
]

wers = [compute_wer(h, r) for h, r in pairs]
avg_wer = np.mean(wers)

print('句子 WER 分布：')
for i, (w, (h, r)) in enumerate(zip(wers, pairs), 1):
    status = '✅' if w == 0 else '❌'
    print(f'  句{i:2d} {status}  WER={w:.3f}  ref_len={len(r)}')
print(f'\n平均 WER = {avg_wer:.4f} = {avg_wer*100:.2f}%')

## 本课小结

本节实现了 `compute_wer`，它通过 Levenshtein 动态规划计算词级编辑距离，输出 `(S+D+I)/N` 浮点 WER 值。该指标将贯穿 `aurora.speech` 模块的全部训练与评测循环。下一节（M3-S2）进入 CTC 损失：理解如何把帧序列对齐到变长词序列，而无需逐帧标注。